# N071 · 树的自底向上聚合

**目标：** 从子树返回最小充分信息而非重复遍历。

**先修：** N069, N063。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 高度/深度；平衡；直径；后序；返回值与全局答案。

**配套讲解来源：** [同名逐章意见](../../comment/071_tree_bottom_up_aggregation.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章要回答三个长得很像的问题：一棵二叉树有多深？它平不平衡？它的直径（最长路径）有多长？

这三个问题有一个共同的坑：如果你直接照着定义写，比如"求节点 1 的高度，就先求它的左子树高度、再求右子树高度"，每个节点都可能把整棵子树重新遍历一遍，最坏会变成 O(n²)。本章的核心思路是"自底向上聚合"：从叶子开始往上算，每个节点只从左右孩子那里各拿一个小数字（子树高度），自己加工一下（取最大值加一），再把结果交给父节点。也就是说，每个子树被压缩成父节点需要的最少信息，整棵树只扫一遍。

举一个具体到数字的例子。输入这棵树（层序序列 `[1,2,3,4,5]`）：

```
        1
       / \
      2   3
     / \
    4   5
```

输出三个答案：最大深度是 3（路径 1→2→4，途经 3 个节点）；它是平衡的（节点 1 左右高度差 2-1=1，节点 2 左右差 1-1=0，都没超过 1）；直径是 3（边数，路径 4→2→1→3，一共 3 条边）。为什么直径是 3 而不是 4？因为直径按"边"计数，这条最长路径虽然经过 4 个节点，但只有 3 条边——这是本章最容易踩的坑，后面会反复强调。

## 2. 基础知识：先读懂这些词

- **二叉树节点（TreeNode）**：一个对象，存三个东西——值 `val`、左孩子指针 `left`、右孩子指针 `right`。孩子不存在时指针是 `None`。它是本章所有结构的基本积木。
- **层序建树（tree_from_level）**：给定一个按"从上到下、从左到右"排列的列表（空位用 `None` 占位），把它还原成真正的树对象。比如 `[1,2,3,4,5]` 就还原出上面那棵树。我们测试时都用它来快速造树。
- **高度（height）/ 深度（depth）**：本章约定"空树高度为 0，一个孤零零的节点高度为 1"。一个节点的高度 = 它往下数到最深的叶子要走几步（按节点数）。"深度"从根往下数，"高度"从叶子往上数，本章代码里实际只用高度。
- **后序遍历（postorder）**：先处理左子树、再处理右子树、最后处理节点本身的顺序。为什么需要它？因为自底向上聚合必须保证"算父节点时，两个孩子已经算完了"，后序正好提供这个保证。
- **平衡（balanced）**：每个节点的左右子树高度差不超过 1。注意是"每个节点"都要检查，不是只检查根。
- **直径（diameter）**：树上任意两点之间最长路径的边数。单节点树直径是 0（一个点没有边）。它和高度的计数单位不同：高度数的是节点个数，直径数的是边条数。
- **经过某节点的最长路径**：一条路径一定有唯一一个"最高点"（离根最近的点）。这条路径从最高点的左子树某叶子上来、穿过最高点、再下到右子树某叶子，所以长度 = 左子树高度 + 右子树高度（按边算正好是这两个数相加）。
- **显式栈代替递归**：本章用迭代器 + 字典模拟后序，而不是写递归函数。原因是真实场景中树可能很深（比如一条 10 万层的链），Python 递归默认深度约 1000 层，会直接爆栈（RecursionError），显式栈能避免这个问题。

## 3. 思路推导：从小例子开始

我们以那棵 5 节点小树为例，把整条思路走一遍。

- **Step 1**：确定每个节点要向父节点汇报什么信息。求高度只需要一个数：我的子树高度。这个信息足够父节点算它自己的高度，这就是"最小充分信息"。
- **Step 2**：确定处理顺序。用后序遍历，保证处理任何节点时它的左右孩子已经汇报完毕。这棵树的后序序列是 4, 5, 2, 3, 1。
- **Step 3**：定递推式。`heights[None] = 0` 是地基（空树高度 0）；每个节点的高度 = `1 + max(左孩子高度, 右孩子高度)`。
- **Step 4**：手算一轮。处理 4：左右都是 None，高度 = 1+max(0,0) = 1。处理 5：同理 = 1。处理 2：左孩子 4 高 1，右孩子 5 高 1，高度 = 1+max(1,1) = 2。处理 3：左右是 None，高度 = 1。处理 1：左 2 高 2，右 3 高 1，高度 = 1+max(2,1) = 3。最大深度就是根的高度 3。
- **Step 5**：平衡检查顺手做。在 Step 4 每一步算高度时，顺便看一眼左右高度差的绝对值：节点 4 差 0、节点 5 差 0、节点 2 差 0、节点 3 差 0、节点 1 差 |2-1|=1。全部 ≤ 1，所以这棵树平衡。
- **Step 6**：直径也在同一趟里做。每到一个节点，算"经过我的最长路径" = 左高 + 右高，拿它去刷新全局最大值。节点 4 给 0+0=0，节点 5 给 0，节点 2 给 1+1=2，节点 3 给 0，节点 1 给 2+1=3。全局最大是 3，这就是直径。

notebook 的 “运行示例”部分 把 Step 4 到 Step 6 的过程列成了表，你可以逐行核对上面的手算：

| 节点 | 左高 | 右高 | 本树高 | 经过本点的直径候选 |
|------|------|------|--------|--------------------|
| 4    | 0    | 0    | 1      | 0                  |
| 5    | 0    | 0    | 1      | 0                  |
| 2    | 1    | 1    | 2      | 2                  |
| 3    | 0    | 0    | 1      | 0                  |
| 1    | 2    | 1    | 3      | 3                  |

- **Step 7**：如果输入是空树（`root is None`），字典里 `heights[None]=0` 已经垫底，直接返回 0，无需特判——这是"把边界写进初始状态"的技巧。

## 4. 核心代码：max_depth

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def max_depth(root):
    heights = {None: 0}
    for node in _postorder_nodes(root):
        heights[node] = 1 + max(heights[node.left], heights[node.right])
    return heights[root]
```

### 逐段读懂代码

“分段实现”部分 里一共有四个部分，我们逐个看。

### 1. 节点定义与建树工具

（对应程序见下方分段实现与完整代码。）

`TreeNode` 是标准的树节点。`tree_from_level` 用队列做广度优先建树：队列里存"还没挂孩子的节点"，下标 `i` 扫过输入列表；每弹出一个节点，就从列表里按顺序取两个值挂成它的左右孩子（值是 `None` 就跳过不挂），刚挂上的新节点进队等下一轮挂孩子。第一行 `if not values or values[0] is None: return None` 挡住空输入和根就是 None 的情况，返回空树。

### 2. 显式后序遍历器

```python
def _postorder_nodes(root):
    stack = [(root, False)]
    while stack:
        node, done = stack.pop()
        if not node:
            continue
        if done:
            yield node
        else:
            stack.append((node, True))
            stack.append((node.right, False))
            stack.append((node.left, False))
```

这是用栈模拟后序的核心技巧。栈里每项是 `(节点, 是否已展开过)`。第一次弹出某节点时（`done=False`），说明它的孩子还没处理，于是按"自己（标记 True）、右孩子、左孩子"的顺序压回栈。栈是后进先出的，所以下次先处理左孩子、再右孩子，最后轮到标记 True 的自己——这时（`done=True`）才真正把它交出去（`yield`）。`if not node: continue` 把压进去的 `None` 孩子直接跳过。这样产出的顺序就是标准的左、右、根。

### 3. 三个主函数

```python
def max_depth(root):
    heights = {None: 0}
    for node in _postorder_nodes(root):
        heights[node] = 1 + max(heights[node.left], heights[node.right])
    return heights[root]
```

`heights` 是一张"节点 → 子树高度"的字典，开头先写死 `heights[None]=0`，这就是空树的地基，后面所有叶子的计算都从它起步。循环按后序拿到每个节点，此刻它的两个孩子必然已在字典里，于是 `1+max(左右)` 一行完成聚合。最后根节点的高度就是最大深度。如果 `root` 本身是 `None`，循环体一次都不执行（栈弹出 None 直接 continue），返回 `heights[None]=0`，边界自动成立。

```python
def is_balanced(root):
    heights = {None: 0}
    for node in _postorder_nodes(root):
        left, right = (heights[node.left], heights[node.right])
        if abs(left - right) > 1:
            return False
        heights[node] = 1 + max(left, right)
    return True
```

它和 `max_depth` 骨架一样，只多了一件事：算高度之前先看 `abs(left-right)`（左右高度差的绝对值），一旦超过 1 就立刻 `return False` 提前退出，剩下的子树不用再看了。能坚持扫完所有节点，说明每个节点都满足定义，返回 `True`。空树时循环不执行，直接返回 `True`——空树被视为平衡。

```python
def diameter(root):
    heights = {None: 0}
    best = 0
    for node in _postorder_nodes(root):
        left, right = (heights[node.left], heights[node.right])
        best = max(best, left + right)
        heights[node] = 1 + max(left, right)
    return best
```

依然是同一副骨架，这次多维护一个全局变量 `best`（目前见过的最长路径）。每到一个节点，"经过我的最长路径"是 `left+right`（从左子树最深处上来、穿过自己、下到右子树最深处，按边算正好是两个高度相加），用它刷新 `best`。`best=0` 的初值对应单节点树：没有任何边，直径为 0。注意返回的是 `best` 而不是任何高度——直径不一定经过根，所以不能只看根节点那一处。

三个函数长得几乎一样不是巧合：它们共享"后序 + 高度字典"这套骨架，只是每个节点上叠的"额外动作"不同（什么都不做 / 查平衡 / 刷直径）。学会这副骨架，你可以自己往上叠新统计量。

## 5. 分段实现：按顺序运行

**本章接口：** `max_depth(root)`、`is_balanced(root)`、`diameter(root)`

### TreeNode

In [1]:
class TreeNode:

    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = (val, left, right)

### tree_from_level

In [2]:
def tree_from_level(values):
    """Construct a finite binary tree from level order with None placeholders."""
    from collections import deque
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values):
            if values[i] is not None:
                node.right = TreeNode(values[i])
                queue.append(node.right)
            i += 1
    return root

### _postorder_nodes

In [3]:
def _postorder_nodes(root):
    stack = [(root, False)]
    while stack:
        node, done = stack.pop()
        if not node:
            continue
        if done:
            yield node
        else:
            stack.append((node, True))
            stack.append((node.right, False))
            stack.append((node.left, False))

### max_depth

In [4]:
def max_depth(root):
    heights = {None: 0}
    for node in _postorder_nodes(root):
        heights[node] = 1 + max(heights[node.left], heights[node.right])
    return heights[root]

### is_balanced

In [5]:
def is_balanced(root):
    heights = {None: 0}
    for node in _postorder_nodes(root):
        left, right = (heights[node.left], heights[node.right])
        if abs(left - right) > 1:
            return False
        heights[node] = 1 + max(left, right)
    return True

### diameter

In [6]:
def diameter(root):
    heights = {None: 0}
    best = 0
    for node in _postorder_nodes(root):
        left, right = (heights[node.left], heights[node.right])
        best = max(best, left + right)
        heights[node] = 1 + max(left, right)
    return best

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [7]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

r=tree_from_level([1,2,3,4,5]); heights={None:0}; rows=[]
for node in _postorder_nodes(r):
    left,right=heights[node.left],heights[node.right]
    heights[node]=1+max(left,right); rows.append((node.val,left,right,heights[node],left+right))
show_table(['节点','左高','右高','本树高','经过本点的直径候选'],rows)

节点,左高,右高,本树高,经过本点的直径候选
4,0,0,1,0
5,0,0,1,0
2,1,1,2,2
3,0,0,1,0
1,2,1,3,3


## 7. 正确性、适用条件与复杂度

任意简单树路径有唯一最高节点；其两段分别落在该节点左右子树。每个最高节点的最优两段就是最大向下高度，因此取全局最大完备。

**代价：** O(n)时间；显式后序与高度字典O(n)辅助空间，避免深树递归溢出。

### 展开理解

**为什么直径算法是对的？** 任选树上的一条路径，它一定有一个唯一点是"最高"的（离根最近，路径先从这一点往下走到一端，再原路折回、往下走到另一端）。这条路径被最高点切成两段：一段完全落在它的左子树里，另一段完全落在右子树里。要让总长度最大，两段就应该各自取到"从最高点往下能走的最远距离"，也就是左右子树的最大向下高度——恰好就是代码里的 `left+right`。而任何路径的最高点必然是树中某一个实际节点，我们对每个节点都算了这个候选值并取全局最大，所以不会漏掉最优的那条路径。

**为什么高度递推是对的？** 空树高度 0 写死在字典里；每个节点的高度完全由两个孩子的高度决定（取大者加一），后序又保证了取的时候孩子一定算完了，所以从叶子一路推到根，每一步的值都是可靠的。

**为什么平衡判断是对的？** "平衡"的定义要求每个节点左右高度差 ≤ 1。我们的循环恰好逐个节点检查了这个差值，一个不漏；而且检查用的是刚算好的、正确的子树高度，不是过期数据。

**复杂度**：每个节点进栈出栈各常数次、字典读写各常数次，所以三个函数都是 O(n) 时间，n 是节点数。n=10⁵ 时大约就是几十万次简单操作，一眨眼算完。辅助空间也是 O(n)：栈和高度字典最坏都和节点数同阶（比如一条链，栈里同时压着整条链）。这个 O(n) 空间换来的是"绝不递归爆栈"——一条 10⁵ 层的链直接递归必挂，这里稳稳跑完。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言分三组：

- `assert diameter(TreeNode(1))==0`：单节点树，是"直径按边计数"的直接检验——一个点没有边，答案必须是 0 而不是 1。
- `r=TreeNode(1,TreeNode(2,TreeNode(3))); assert diameter(r)==2 and not is_balanced(r)`：一条向左歪的链 1→2→3。直径是 2（1 到 3 有两条边），同时这条链严重不平衡（根的左子树高 2、右子树高 0，差 2），所以 `is_balanced` 必须返回假。一条链同时测了两个函数的极端行为。
- `assert max_depth(None)==0 and is_balanced(None)`：空树边界。空树深度为 0、视为平衡，检验的是我们"把边界写进初始状态"的设计有没有真的兜住 `root=None`。
- 最后一段随机对拍测试：用随机数生成器造出 1 到 29 个节点的随机树，再用一种完全不同的笨办法求最长路径（从每个节点出发做一次广度优先搜索，记录能走到的最远步数），然后断言本章的 `diameter` 和笨办法答案一致。因为两种方法思路完全不同，它们同时出错的可能性极小，这比手写几个固定用例的说服力强得多。29 棵树全部通过后打印 `N071: 所有本章断言通过`。

In [8]:
assert diameter(TreeNode(1)) == 0

r = TreeNode(1, TreeNode(2, TreeNode(3)))

assert diameter(r) == 2 and (not is_balanced(r))

assert max_depth(None) == 0 and is_balanced(None)

from collections import deque

from random import Random

rng = Random(71)

for n in range(1, 30):
    nodes = [TreeNode(i) for i in range(n)]
    slots = [(nodes[0], 'left'), (nodes[0], 'right')]
    adj = {x: [] for x in nodes}
    for node in nodes[1:]:
        parent, side = slots.pop(rng.randrange(len(slots)))
        setattr(parent, side, node)
        slots.extend([(node, 'left'), (node, 'right')])
        adj[parent].append(node)
        adj[node].append(parent)
    longest = 0
    for start in nodes:
        q = deque([(start, 0)])
        seen = {start}
        while q:
            u, d = q.popleft()
            longest = max(longest, d)
            for v in adj[u]:
                if v not in seen:
                    seen.add(v)
                    q.append((v, d + 1))
    assert diameter(nodes[0]) == longest

print('N071: 所有本章断言通过')

N071: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 将平衡判断从重复高度计算改为一次遍历。

**练习 2：** 区分按边计与按点计直径。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（把平衡判断改成一次遍历）**：经典 LeetCode 110 有一种朴素写法——对每个节点都重新调一次求高度函数，那样高层节点会反复扫底层，代价是 O(n²)。提示：回到本章骨架，思考"我向父节点汇报什么"，除了高度，能不能顺便汇报"我这棵子树已经不平衡了"这个坏消息？常见做法是约定"返回 -1 表示子树已失衡"，父节点看到任何一边是 -1 就直接一路返回 -1，不再计算。先写一个 5 节点的不平衡小例子手算一遍再动手。
- **练习 2（区分按边计与按点计直径）**：本章 `diameter` 返回边数（单节点为 0）。提示：按点计数的直径，就是路径上的节点个数，数值上等于"边数直径 + 1"。拿本章的 5 节点例子验证：边计直径是 3（4→2→1→3），点计直径是 4（四个节点）。想清楚"经过某节点的最长路径按点计 = left+right+1（加上自己这个点）"之后，改动一行递推就够了。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [9]:
# 完整实现：本单元格不依赖其他单元格。
class TreeNode:

    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = (val, left, right)

def tree_from_level(values):
    """Construct a finite binary tree from level order with None placeholders."""
    from collections import deque
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values):
            if values[i] is not None:
                node.right = TreeNode(values[i])
                queue.append(node.right)
            i += 1
    return root

def _postorder_nodes(root):
    stack = [(root, False)]
    while stack:
        node, done = stack.pop()
        if not node:
            continue
        if done:
            yield node
        else:
            stack.append((node, True))
            stack.append((node.right, False))
            stack.append((node.left, False))

def max_depth(root):
    heights = {None: 0}
    for node in _postorder_nodes(root):
        heights[node] = 1 + max(heights[node.left], heights[node.right])
    return heights[root]

def is_balanced(root):
    heights = {None: 0}
    for node in _postorder_nodes(root):
        left, right = (heights[node.left], heights[node.right])
        if abs(left - right) > 1:
            return False
        heights[node] = 1 + max(left, right)
    return True

def diameter(root):
    heights = {None: 0}
    best = 0
    for node in _postorder_nodes(root):
        left, right = (heights[node.left], heights[node.right])
        best = max(best, left + right)
        heights[node] = 1 + max(left, right)
    return best

# 示例与回归测试
assert diameter(TreeNode(1)) == 0

r = TreeNode(1, TreeNode(2, TreeNode(3)))

assert diameter(r) == 2 and (not is_balanced(r))

assert max_depth(None) == 0 and is_balanced(None)

from collections import deque

from random import Random

rng = Random(71)

for n in range(1, 30):
    nodes = [TreeNode(i) for i in range(n)]
    slots = [(nodes[0], 'left'), (nodes[0], 'right')]
    adj = {x: [] for x in nodes}
    for node in nodes[1:]:
        parent, side = slots.pop(rng.randrange(len(slots)))
        setattr(parent, side, node)
        slots.extend([(node, 'left'), (node, 'right')])
        adj[parent].append(node)
        adj[node].append(parent)
    longest = 0
    for start in nodes:
        q = deque([(start, 0)])
        seen = {start}
        while q:
            u, d = q.popleft()
            longest = max(longest, d)
            for v in adj[u]:
                if v not in seen:
                    seen.add(v)
                    q.append((v, d + 1))
    assert diameter(nodes[0]) == longest

print('N071: 所有本章断言通过')

N071: 所有本章断言通过


## 11. 代表题与迁移

- **104. Maximum Depth of Binary Tree**：求最大深度，直接就是本章 `max_depth`，练的是"后序 + 高度字典"这副最基础的骨架。
- **110. Balanced Binary Tree**：判断平衡树，对应 `is_balanced`，练的是"在聚合高度的同时顺手做多一项检查"，也正是练习 1 的原题。
- **543. Diameter of Binary Tree**：二叉树直径，对应 `diameter`，练的是"全局答案与返回值分离"——每个节点返回高度给父节点用，直径候选值另外拿全局变量记。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。